<div dir="rtl">

# 🚀 سیستم کامل — همه‌چیز با یک «Run all»

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ParsaVictor/visual-intelligence-engine/blob/main/notebooks/0_run_everything.ipynb)

این نوت‌بوک کل سیستم را از اول تا آخر اجرا می‌کند:

| مرحله | چه می‌کند |
|:--|:--|
| ۱. آماده‌سازی | Drive را وصل می‌کند، GPU را پیدا می‌کند، بسته را نصب می‌کند |
| ۲. ایندکس عکس‌ها | **بار اول همه‌ی دیتاست را می‌خواند؛ از بار دوم فقط عکس‌های جدید یا تغییرکرده را** |
| ۳. جستجو | چهره، حیوان، غذا و متن آزاد، با خروجی تصویری آماده‌ی ارائه |
| ۴. ویدیو | تحلیل ویدیوها، گزارش HTML و **ویدیوی خروجی با کادر چهره، نام افراد و خط زمانی** |
| ۵. خلاصه | همه‌ی خروجی‌ها در Drive، با زمان هر مرحله |

## نحوه‌ی اجرا (از گوشی هم می‌شود)

1. منوی **Runtime → Change runtime type → T4 GPU**
2. عکس‌ها را در Drive در پوشه‌ی `MyDrive/vie/dataset` بگذارید (پوشه یا یک فایل `zip`). زیرپوشه هم اشکالی ندارد.
3. **Runtime → Run all**. اگر Drive اجازه خواست، تأیید کنید.

اگر پوشه‌ی دیتاست خالی باشد، نوت‌بوک یک مجموعه‌ی کوچک عمومی دانلود می‌کند تا همه‌ی مراحل باز هم اجرا شوند.

## چرا بار دوم خیلی سریع است

ایندکس روی **Google Drive** ذخیره می‌شود، نه روی دیسک موقت Colab که با هر بار بسته شدن پاک می‌شود.
هر عکس با اندازه و زمان تغییرش شناخته می‌شود؛ عکسی که عوض نشده **اصلاً باز نمی‌شود**،
و اگر هیچ عکس جدیدی نباشد **هیچ مدلی هم بار نمی‌شود**. اگر وسط کار قطع شود، کار انجام‌شده هر چند دقیقه در Drive ذخیره شده و ادامه‌ی کار از همان‌جا شروع می‌شود.

## دقت

| بخش | مدل | چرا |
|:--|:--|:--|
| چهره | ArcFace (InsightFace `buffalo_l`) **+ میانگین با تصویر آینه‌ای** | در مقایسه‌ی مستقیم با SFace (ریپازیتوری گیت دانشجویی)، روی چهره‌های با زاویه‌ی سخت ٪۹۷٫۶ در برابر ٪۸۳٫۶ و بدون هیچ پذیرش اشتباه. جزئیات در `results/face_benchmark.json` |
| جستجوی چهره با چند عکس | بیشترین شباهت بین عکس‌های پرس‌وجو | در گیت دانشجویی از ٪۸۶ به ٪۹۶ رسید |
| متن، حیوان، غذا | SigLIP so400m روی GPU | دقیق‌ترین مدل این خانواده؛ روی CPU مدل سبک‌تر انتخاب می‌شود |

</div>

<div dir="rtl">

## ⚙️ تنظیمات

**فقط همین سلول را تغییر دهید.** بقیه‌ی سلول‌ها خودکارند.

</div>

In [ ]:
# ═══════════════════ تنظیمات ═══════════════════
DRIVE_ROOT   = "/content/drive/MyDrive/vie"   # همه‌چیز اینجا می‌ماند و بین اجراها حفظ می‌شود
DATASET      = f"{DRIVE_ROOT}/dataset"        # پوشه‌ی عکس‌ها، یا مسیر یک فایل .zip
VIDEOS       = f"{DRIVE_ROOT}/videos"         # ویدیوها (اختیاری)
FACE_QUERIES = f"{DRIVE_ROOT}/queries"        # queries/<نام فرد>/*.jpg  ← ۱ تا ۳ عکس از هر نفر
KNOWN_PEOPLE = f"{DRIVE_ROOT}/people"         # people/<نام>/*.jpg  ← برای نام‌گذاری افراد در ویدیو

ANIMAL_QUERIES = ["cat", "dog", "bird", "horse", "zebra"]
FOOD_QUERIES   = ["pizza", "kebab", "salad", "cake"]
TEXT_QUERIES   = ["a man with a blue backpack", "people smiling at a party",
                  "a sunset over the sea", "a person reading a book"]
VIDEO_QUERIES  = ["people talking", "a person walking in a corridor"]
# جستجوی مفهوم: یک توضیح، همه‌ی عکس‌ها و لحظه‌های ویدیو که آن را نشان می‌دهند.
# با QUALITY = "multilingual" (یا "auto" روی GPU) فارسی هم می‌شود نوشت.
CONCEPT_QUERIES = ["a theatre performance on stage", "people in a classroom",
                   "a celebration with friends", "تئاتر", "کلاس درس"]

QUALITY        = "auto"   # auto | best | multilingual | fast   (auto: بهترین مدل روی GPU، سبک روی CPU)
RENDER_VIDEOS  = True     # ساخت ویدیوی خروجی با کادر و نام افراد
MAX_RENDER_SECONDS = 180  # سقف طول هر ویدیوی خروجی، برای صرفه‌جویی در زمان
DEMO_IF_EMPTY  = True     # اگر دیتاست خالی بود، داده‌ی نمونه‌ی عمومی دانلود شود
REPO_REF       = "claude/image-processing-optimization-7abxze"   # شاخه، برچسب یا commit؛ اگر نبود main
# نسخه‌ی v0.5.0 (پیش از جستجوی مفهوم و SigLIP 2): REPO_REF = "c3ef80c8e11f722908f1463eb7e950cd029cd7dd"

<div dir="rtl">

## ۱. آماده‌سازی

بار اول ۳ تا ۵ دقیقه طول می‌کشد. هر خطایی که پیش بیاید با پیام فارسی نشان داده می‌شود و اجرا متوقف نمی‌شود.

</div>

In [ ]:
import os, sys, json, time, shutil, sqlite3, subprocess, traceback, contextlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
elif DRIVE_ROOT.startswith("/content/"):
    DRIVE_ROOT = os.path.abspath("vie_drive")               # خارج از Colab: یک پوشه‌ی محلی
    DATASET, VIDEOS = f"{DRIVE_ROOT}/dataset", f"{DRIVE_ROOT}/videos"
    FACE_QUERIES, KNOWN_PEOPLE = f"{DRIVE_ROOT}/queries", f"{DRIVE_ROOT}/people"

GPU = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True)
HAS_GPU = GPU.returncode == 0 and "GPU" in GPU.stdout
print("GPU:", GPU.stdout.strip() if HAS_GPU else "ندارد — روی CPU کندتر اجرا می‌شود "
      "(Runtime → Change runtime type → T4 GPU)")

SRC = Path("/content/visual-intelligence-engine" if IN_COLAB else "vie-src").resolve()
URL = "https://github.com/ParsaVictor/visual-intelligence-engine.git"
def fetch_ref(ref):
    """یک شاخه، برچسب یا شناسه‌ی commit را در SRC می‌گذارد."""
    shutil.rmtree(SRC, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref, URL, str(SRC)]).returncode == 0:
        return True
    # شناسه‌ی commit با clone -b نمی‌آید؛ مستقیم fetch می‌شود.
    shutil.rmtree(SRC, ignore_errors=True)
    SRC.mkdir(parents=True)
    git = lambda *a: subprocess.run(["git", "-C", str(SRC), *a], capture_output=True).returncode == 0
    return git("init", "-q") and git("remote", "add", "origin", URL) and \
        git("fetch", "-q", "--depth", "1", "origin", ref) and git("checkout", "-q", "FETCH_HEAD")

if not fetch_ref(REPO_REF):
    print(f"⚠️ نسخه‌ی {REPO_REF} پیدا نشد؛ از main استفاده می‌شود")
    assert fetch_ref("main"), "دریافت کد از گیت‌هاب ممکن نشد"
print("کد:", subprocess.run(["git", "-C", str(SRC), "log", "-1", "--format=%h %s"],
                            capture_output=True, text=True).stdout.strip())

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-2000:])
    return r.returncode == 0

t0 = time.time()
pip("-e", f"{SRC}[models,video]")
# دقیقاً یکی از دو نسخه‌ی onnxruntime، وگرنه با هم تداخل دارند
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime", "onnxruntime-gpu"],
               capture_output=True)
pip("onnxruntime-gpu" if HAS_GPU else "onnxruntime")
pip("arabic-reshaper", "python-bidi")   # نوشتن درست نام‌های فارسی روی تصویر، اگر Pillow خودش نتواند

# ── بررسی نصب در یک فرایند تازه، تا خطای نصب همین‌جا دیده شود نه وسط کار ──
CHECK = """
import cv2, numpy, onnxruntime, insightface, torch, transformers, av, PIL
assert cv2.imdecode(numpy.frombuffer(cv2.imencode('.png', numpy.zeros((4, 4, 3), 'uint8'))[1], 'uint8'), 1) is not None
print('opencv', cv2.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| onnx', onnxruntime.get_available_providers()[:2])
"""
probe = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
if probe.returncode and "cv2" in probe.stderr:
    # Colab ships opencv-python and insightface brings the headless build; when the
    # two disagree, cv2 breaks. Reinstalling one build cleanly fixes it.
    print("OpenCV ناسازگار بود؛ دوباره نصب می‌شود…")
    pip("--force-reinstall", "--no-deps", "opencv-python-headless")
    probe = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
print(probe.stdout.strip() or probe.stderr[-1500:])
if HAS_GPU and "CUDAExecutionProvider" not in probe.stdout:
    print("⚠️ مدل چهره روی GPU اجرا نمی‌شود (onnxruntime-gpu)؛ روی CPU کار می‌کند ولی کندتر.")

sys.path.insert(0, str(SRC / "src"))   # نصب editable در kernel در حال اجرا بدون این دیده نمی‌شود
import vie
print(f"✅ Visual Intelligence Engine {vie.__version__} آماده شد در {time.time() - t0:.0f} ثانیه")

In [ ]:
# ── ابزارهای کمکی نوت‌بوک ───────────────────────────────────────────
from IPython.display import HTML, Image as IPImage, Video, display

STATUS = []   # خلاصه‌ی نهایی: (مرحله، وضعیت، ثانیه، توضیح)

@contextlib.contextmanager
def stage(title):
    """هر مرحله جدا اجرا می‌شود؛ خطای یک مرحله بقیه را متوقف نمی‌کند."""
    started = time.time()
    display(HTML(f'<div dir="rtl" style="font:600 16px system-ui;margin:18px 0 6px;'
                 f'color:#ffb454">▶ {title}</div>'))
    try:
        yield
        STATUS.append((title, "✅", time.time() - started, ""))
    except Exception as exc:
        STATUS.append((title, "⚠️", time.time() - started, f"{type(exc).__name__}: {exc}"))
        display(HTML(f'<div dir="rtl" style="background:#3a1d1d;color:#ffd7d7;padding:10px 14px;'
                     f'border-radius:8px;font:14px system-ui">⚠️ این مرحله انجام نشد: '
                     f'<code>{type(exc).__name__}: {exc}</code><br>بقیه‌ی مراحل ادامه پیدا می‌کنند.</div>'))
        traceback.print_exc(limit=3)

def say(text):
    display(HTML(f'<div dir="rtl" style="font:15px/1.8 system-ui">{text}</div>'))

def show_image(path, width=1100):
    if Path(path).exists():
        display(IPImage(filename=str(path), width=width))

def show_video(path, width=900):
    path = Path(path)
    if not path.exists():
        return
    if path.stat().st_size < 40e6:
        display(Video(str(path), embed=True, width=width, html_attributes="controls muted"))
    else:
        say(f"ویدیو بزرگ‌تر از آن است که اینجا پخش شود؛ در Drive ببینید: <code>{path}</code>")

<div dir="rtl">

### دیتاست، ایندکس و مدل‌ها

- ایندکس اصلی در `DRIVE_ROOT/index` می‌ماند. هنگام کار یک کپی روی دیسک سریع Colab ساخته می‌شود، چون SQLite روی Drive کند و شکننده است. هر چند دقیقه و در پایان هر مرحله، کپی به Drive برگردانده می‌شود.
- برای هر مدل یک ایندکس جدا ساخته می‌شود، پس عوض کردن `QUALITY` هیچ‌وقت ایندکس را خراب نمی‌کند.
- مدل‌ها یک بار دانلود می‌شوند و در `DRIVE_ROOT/models` می‌مانند.

</div>

In [ ]:
import zipfile, urllib.request

for d in (DRIVE_ROOT, VIDEOS, FACE_QUERIES, KNOWN_PEOPLE, f"{DRIVE_ROOT}/index", f"{DRIVE_ROOT}/outputs"):
    os.makedirs(d, exist_ok=True)
OUT = Path(f"{DRIVE_ROOT}/outputs")
LOCAL = Path("/content/vie_local" if IN_COLAB else "vie_local").resolve()
LOCAL.mkdir(parents=True, exist_ok=True)

# ── دیتاست: پوشه یا zip ──────────────────────────────────────────────
GALLERY = Path(DATASET)
if DATASET.lower().endswith(".zip"):
    GALLERY = LOCAL / "dataset"
    marker = GALLERY / ".extracted"
    zstat = os.stat(DATASET)
    stamp = f"{zstat.st_size}-{zstat.st_mtime_ns}"
    if not marker.exists() or marker.read_text() != stamp:
        print("از حالت فشرده خارج می‌شود…")
        with zipfile.ZipFile(DATASET) as z:
            z.extractall(GALLERY)
            # extractall stamps every file "now", which would make the whole dataset
            # look changed each session. Restoring the archive's own times lets the
            # index skip unchanged photos without even reading them.
            for info in z.infolist():
                target = GALLERY / info.filename
                if not info.is_dir() and target.exists():
                    when = time.mktime(info.date_time + (0, 0, -1))
                    os.utime(target, (when, when))
        marker.write_text(stamp)
GALLERY.mkdir(parents=True, exist_ok=True)

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
def count_images(root):
    return sum(1 for p in Path(root).rglob("*") if p.suffix.lower() in IMAGE_EXT)

# ── داده‌ی نمونه اگر دیتاست خالی است ─────────────────────────────────
RAW = "https://raw.githubusercontent.com"
DEMO_VIDEOS = ["head-pose-face-detection-female-and-male.mp4", "face-demographics-walking-and-pause.mp4",
               "classroom.mp4"]
def fetch(url, dest):
    dest = Path(dest)
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, dest)
    return dest

if DEMO_IF_EMPTY and count_images(GALLERY) == 0:
    say("دیتاست خالی است؛ یک مجموعه‌ی کوچک عمومی (ویدیوهای CC BY 4.0 اینتل و عکس‌های نمونه) دانلود می‌شود.")
    import av
    for name in DEMO_VIDEOS + ["head-pose-face-detection-female.mp4"]:
        fetch(f"{RAW}/intel-iot-devkit/sample-videos/master/{name}", LOCAL / "demo_videos" / name)
    for name in ("dog.jpg", "horses.jpg", "eagle.jpg", "giraffe.jpg", "person.jpg"):
        try:
            fetch(f"{RAW}/pjreddie/darknet/master/data/{name}", GALLERY / "animals" / name)
        except Exception as exc:
            print("skip", name, exc)
    def stills(video, every, prefix):
        with av.open(str(video)) as c:
            nxt = 0.0
            for f in c.decode(video=0):
                if f.time is not None and f.time >= nxt:
                    f.to_image().save(GALLERY / "people" / f"{prefix}_{f.time:06.1f}.jpg", quality=92)
                    nxt += every
    (GALLERY / "people").mkdir(parents=True, exist_ok=True)
    for name, every in (("head-pose-face-detection-female-and-male.mp4", 10),
                        ("face-demographics-walking-and-pause.mp4", 8), ("classroom.mp4", 8)):
        stills(LOCAL / "demo_videos" / name, every, Path(name).stem[:12])
    q = Path(FACE_QUERIES) / "demo_person"
    if not q.exists():
        q.mkdir(parents=True)
        with av.open(str(LOCAL / "demo_videos" / "head-pose-face-detection-female.mp4")) as c:
            for f in c.decode(video=0):
                if f.time and f.time >= 1.0:
                    f.to_image().save(q / "front.jpg"); break
    for name in DEMO_VIDEOS[:2]:
        if not (Path(VIDEOS) / name).exists():
            shutil.copy(LOCAL / "demo_videos" / name, Path(VIDEOS) / name)

print(f"📁 دیتاست: {GALLERY}  —  {count_images(GALLERY):,} عکس")

# ── انتخاب مدل ────────────────────────────────────────────────────────
BEST_MODEL = "google/siglip-so400m-patch14-384"
AUTO_ON_GPU = "multilingual"   # بر اساس scripts/eval_search.py
# best: دقیق‌ترین مدل انگلیسی · multilingual: SigLIP 2، فارسی هم می‌فهمد · fast: مدل سبک برای CPU
MODELS = {"best": BEST_MODEL, "multilingual": "google/siglip2-so400m-patch14-384",
          "fast": "google/siglip-base-patch16-224"}
VLM = MODELS.get(QUALITY) or (MODELS[AUTO_ON_GPU] if HAS_GPU else MODELS["fast"])
slug = VLM.split("/")[-1]
DB_DRIVE = Path(f"{DRIVE_ROOT}/index/index-{slug}.db")
DB_LOCAL = LOCAL / DB_DRIVE.name

def sqlite_copy(src, dst):
    """کپی سازگار یک پایگاه SQLite، حتی وسط کار، و جایگزینی اتمی.

    نسخه‌ی پشتیبان اول روی دیسک محلی ساخته می‌شود، چون SQLite روی Drive کند و
    شکننده است. بعد یک فایل کامل کپی و با یک rename جایگزین می‌شود، پس فایل مقصد
    هیچ‌وقت نیمه‌کاره نمی‌ماند.
    """
    snap = LOCAL / (Path(src).name + ".snapshot")
    snap.unlink(missing_ok=True)
    a, b = sqlite3.connect(src), sqlite3.connect(snap)
    try:
        a.backup(b)
    finally:
        a.close(); b.close()
    tmp = Path(str(dst) + ".part")
    shutil.copyfile(snap, tmp)
    os.replace(tmp, dst)

if DB_DRIVE.exists() and (not DB_LOCAL.exists() or DB_DRIVE.stat().st_mtime > DB_LOCAL.stat().st_mtime):
    sqlite_copy(DB_DRIVE, DB_LOCAL)
    print("♻️ ایندکس قبلی از Drive بازیابی شد")

_last_save = [0.0]
def save_index(force=True):
    if DB_LOCAL.exists() and (force or time.time() - _last_save[0] > 180):
        sqlite_copy(DB_LOCAL, DB_DRIVE)
        _last_save[0] = time.time()

from vie.api import Engine
from vie.logging_setup import configure
configure("WARNING")
engine = Engine.from_overrides({
    "paths": {"gallery": str(GALLERY), "videos": str(VIDEOS), "database": str(DB_LOCAL),
              "model_cache": f"{DRIVE_ROOT}/models"},
    "runtime": {"device": "auto", "low_memory": not HAS_GPU},
    "models": {"vision_language": VLM},
    "indexing": {"batch_size": 16 if HAS_GPU else 4, "workers": 8, "commit_every": 50},
    "video": {"captions": False, "transcript": False},
})
CFG = engine.config
before = engine.counts()
say(f"مدل تصویر-متن: <b>{VLM}</b> · چهره: <b>ArcFace + آینه</b> · "
    f"در ایندکس: <b>{before['images']:,}</b> عکس، <b>{before['faces']:,}</b> چهره، "
    f"<b>{before['videos']}</b> ویدیو")

<div dir="rtl">

## ۲. ایندکس عکس‌ها

- **بار اول** همه‌ی عکس‌ها خوانده می‌شوند: چهره، حیوان و غذا تشخیص داده می‌شوند و بردار تصویر ساخته می‌شود.
- **از بار دوم** فقط عکس‌های جدید یا تغییرکرده خوانده می‌شوند.
- عکسی که از دیتاست پاک شده باشد از ایندکس هم حذف می‌شود. اما اگر ناگهان بیشتر دیتاست غیب شده باشد (مثلاً Drive هنوز وصل نشده)، سیستم چیزی را پاک نمی‌کند.

</div>

In [ ]:
with stage("ایندکس عکس‌ها"):
    from tqdm.auto import tqdm
    bar = tqdm(desc="عکس‌های جدید", unit="img")
    def on_progress(stats):
        bar.n = stats.indexed; bar.refresh()
        save_index(force=False)            # هر ۳ دقیقه یک نسخه‌ی امن در Drive
    first = before["images"] == 0
    INDEX_REPORT = engine.index_images(progress=on_progress)
    bar.close()
    save_index()
    r = INDEX_REPORT
    if not r.pruning:
        say("⚠️ بیشتر عکس‌های ایندکس‌شده در پوشه پیدا نشدند، پس هیچ چیز از ایندکس حذف نشد. "
            "مسیر DATASET یا اتصال Drive را بررسی کنید.")
    after = engine.counts()
    kind = "اولین ایندکس کامل" if first else "به‌روزرسانی افزایشی"
    say(f"<b>{kind}</b>: {r.scanned:,} عکس پیدا شد · <b>{r.indexed:,}</b> عکس تازه پردازش شد · "
        f"<b>{r.skipped:,}</b> عکس بدون تغییر بود و اصلاً باز نشد · {r.pruned} حذف · "
        f"{r.failed} خطا · {r.seconds:.1f} ثانیه"
        + (f" · {r.images_per_second:.1f} عکس در ثانیه" if r.indexed else ""))
    say(f"اکنون در ایندکس: {after['images']:,} عکس · {after['faces']:,} چهره · "
        f"{after['animals']:,} عکس با حیوان · {after['food']:,} عکس با غذا")

In [ ]:
with stage("اثبات افزایشی بودن: اجرای دوباره"):
    again = engine.index_images()
    say(f"اجرای دوباره روی همان دیتاست: <b>{again.indexed}</b> عکس پردازش شد، "
        f"{again.skipped:,} عکس رد شد، در <b>{again.seconds:.2f}</b> ثانیه. "
        "عکس جدید که اضافه کنید، فقط همان پردازش می‌شود.")

<div dir="rtl">

## ۳. جستجو، با خروجی آماده‌ی ارائه

برای هر جستجو یک تصویر (PNG) و یک گزارش HTML در `DRIVE_ROOT/outputs` ساخته می‌شود.

- **چهره:** برای هر نفر یک پوشه در `queries` بسازید و ۱ تا ۳ عکس از او بگذارید. سیستم با همه‌ی عکس‌ها جستجو می‌کند و برای هر چهره در آرشیو بهترین شباهت را نگه می‌دارد. اگر پوشه خالی باشد، پرتکرارترین چهره‌ی آرشیو انتخاب می‌شود.

</div>

In [ ]:
CARDS = OUT / "cards"; REPORTS = OUT / "reports"
CARDS.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)
SHOWN = []

def present(result, name):
    card = engine.save_card(result, CARDS / f"{name}.png")
    engine.save_html(result, REPORTS / f"{name}.html")
    SHOWN.append(card)
    best = f"{result.matches[0].display_score * 100:.1f}٪" if result.matches else "—"
    say(f"«{result.query}» → <b>{len(result.matches)}</b> نتیجه · بهترین {best} · "
        f"{result.milliseconds:.0f} میلی‌ثانیه")
    show_image(card)

def safe_name(text):
    return "".join(c if c.isalnum() else "-" for c in text)[:40].strip("-")

with stage("جستجوی چهره"):
    people = [p for p in sorted(Path(FACE_QUERIES).iterdir()) if p.is_dir()]
    jobs = [(p.name, sorted(x for x in p.iterdir() if x.suffix.lower() in IMAGE_EXT)[:5]) for p in people]
    jobs = [(n, imgs) for n, imgs in jobs if imgs]
    if not jobs:
        # چهره‌ی پرتکرار بریده می‌شود، چون در عکس گروهی چهره‌ی بزرگ‌تر شاید کس دیگری باشد
        auto = engine.most_photographed_face(crop_to=OUT / "queries" / "most-seen-person.jpg")
        jobs = [("most-seen-person", [auto])] if auto else []
    if not jobs:
        say("هیچ چهره‌ای در ایندکس نیست.")
    for name, images in jobs:
        present(engine.search_face(images, top_k=12), f"face-{safe_name(name)}")

In [ ]:
with stage("جستجوی حیوان"):
    for q in ANIMAL_QUERIES:
        result = engine.search_animal(q, top_k=8)
        if result.matches:
            present(result, f"animal-{safe_name(q)}")
        else:
            say(f"«{q}»: در آرشیو پیدا نشد")

In [ ]:
with stage("جستجوی غذا"):
    for q in FOOD_QUERIES:
        result = engine.search_food(q, top_k=8)
        if result.matches:
            present(result, f"food-{safe_name(q)}")
        else:
            say(f"«{q}»: در آرشیو پیدا نشد")

In [ ]:
with stage("جستجوی متنی آزاد"):
    for q in TEXT_QUERIES:
        present(engine.search_text(q, top_k=8), f"text-{safe_name(q)}")

<div dir="rtl">

## ۴. ویدیو: تحلیل، گزارش و ویدیوی خروجی

فقط ویدیوهای **جدید یا تغییرکرده** پردازش می‌شوند. برای هر کدام این خروجی‌ها ساخته می‌شود:

- **گزارش HTML:** نماها، موضوع، افراد، حیوان و غذا روی خط زمانی، و خلاصه به انگلیسی و فارسی.
- **ویدیوی خروجی MP4:** کادر هر چهره، نام فرد (اگر در `people/<نام>/` عکس مرجع گذاشته باشید)، برچسب نما و خط زمانی حضور هر نفر.
- **کارت افراد:** چه کسی، چقدر و از چه زمانی در ویدیو بوده است.

</div>

In [ ]:
VID_OUT = OUT / "videos"; VID_OUT.mkdir(exist_ok=True)
RENDERED = []

with stage("ویدیوها"):
    from vie.report.video import write_video_report
    from vie.video.render import render_people_card
    known = engine.known_people(KNOWN_PEOPLE)
    if known:
        say("افراد مرجع برای نام‌گذاری: " + "، ".join(f"{k} ({len(v)} عکس)" for k, v in known.items()))

    def after(key, path, analysis):
        name = safe_name(Path(key).stem)
        write_video_report(analysis, VID_OUT / f"{name}.html")
        say(f"🎬 <b>{key}</b>: {analysis.summary.get('fa') or analysis.summary.get('en', '')}")
        if RENDER_VIDEOS:
            result = engine.render_video(path, VID_OUT / f"{name}.mp4", known=known or None,
                                         analysis=analysis, title=Path(key).stem[:40],
                                         max_seconds=MAX_RENDER_SECONDS,
                                         detect_every=1 if HAS_GPU else 3)
            card = VID_OUT / f"{name}-people.png"
            render_people_card(result, title=Path(key).stem[:40]).save(card)
            RENDERED.append((result.path, card))
            say(f"ویدیوی خروجی: {result.frames:,} فریم با سرعت {result.fps_processed:.1f} فریم در ثانیه · "
                + "، ".join(f"{p.label}: {p.seconds:.0f} ثانیه" for p in result.people[:6]))
        save_index()

    vstats = engine.index_videos(on_analysed=after)
    save_index()
    say(f"ویدیوها: {vstats.scanned} پیدا شد · <b>{vstats.indexed}</b> ویدیوی تازه تحلیل شد · "
        f"{vstats.skipped} بدون تغییر رد شد · {vstats.failed} خطا")
    if not RENDERED:
        done = sorted(VID_OUT.glob("*.mp4"))
        RENDERED = [(p, p.with_name(p.stem + "-people.png")) for p in done]
        if done:
            say("ویدیوی تازه‌ای نبود؛ خروجی‌های قبلی نمایش داده می‌شوند.")
    for video, card in RENDERED[:3]:
        show_video(video)
        show_image(card)

In [ ]:
with stage("جستجو در ویدیوها"):
    from vie.video.understand import format_clock
    if engine.counts()["videos"] == 0:
        say("ویدیویی در ایندکس نیست.")
    for q in VIDEO_QUERIES:
        moments = engine.search_videos(q, top_k=5)
        rows = "".join(f"<tr><td>{m.video_key}</td><td>{format_clock(m.start)}–{format_clock(m.end)}</td>"
                       f"<td>{m.score:.3f}</td><td>{m.kind}</td></tr>" for m in moments)
        say(f"«{q}»" + (f'<table style="direction:ltr">{rows}</table>' if rows else ": چیزی پیدا نشد"))

<div dir="rtl">

### جستجوی مفهوم در کل آرشیو

یک توضیح، مثلاً «a theatre performance» یا با مدل چندزبانه «تئاتر». جواب شامل **همه‌ی عکس‌ها و لحظه‌های ویدیو** است که آن مفهوم را نشان می‌دهند، در یک فهرست مرتب. روی کاشی هر لحظه‌ی ویدیو، زمانش نوشته شده است. این جستجو در ایندکس انجام می‌شود و هیچ عکس یا ویدیویی باز نمی‌شود، به جز همان چند فریمی که روی کارت نشان داده می‌شوند.

</div>

In [ ]:
with stage("جستجوی مفهوم در عکس‌ها و ویدیوها"):
    from vie.vlm import has_right_to_left, is_multilingual
    for q in CONCEPT_QUERIES:
        if has_right_to_left(q) and not is_multilingual(VLM):
            say(f"«{q}» رد شد: مدل فعلی فقط انگلیسی می‌فهمد. برای فارسی QUALITY = \"multilingual\"")
            continue
        result = engine.search_everything(q, top_k=12, frames_dir=OUT / "frames")
        present(result, f"concept-{safe_name(q)}")

<div dir="rtl">

## ۵. خلاصه

</div>

In [ ]:
save_index()
rows = "".join(f"<tr><td>{s}</td><td>{t}</td><td>{sec:.1f}s</td><td style='direction:ltr'>{note[:160]}</td></tr>"
               for t, s, sec, note in STATUS)
final = engine.counts()
say(f"""<table>{rows}</table><br>
<b>در ایندکس:</b> {final['images']:,} عکس · {final['faces']:,} چهره · {final['videos']} ویدیو<br>
<b>خروجی‌ها در Drive:</b> <code>{OUT}</code> — {len(list(CARDS.glob('*.png')))} کارت جستجو،
{len(list(VID_OUT.glob('*.mp4')))} ویدیوی خروجی، گزارش‌های HTML<br>
<b>دفعه‌ی بعد:</b> عکس یا ویدیوی جدید را به همان پوشه‌ها اضافه کنید و دوباره Run all بزنید؛ فقط موارد جدید پردازش می‌شوند.""")